In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/krupalpatel07/arm-holdings/ARM.csv


In [2]:

# ==========================================================
# 1. IMPORTS
# ==========================================================



import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "iframe"

from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

from IPython.display import HTML, display

In [3]:
# ==========================================================
# 2. LOAD DATA
# ==========================================================

file_path = "/kaggle/input/datasets/krupalpatel07/arm-holdings/ARM.csv"

df = pd.read_csv(file_path)

df.columns = [c.lower() for c in df.columns]

df["date"] = pd.to_datetime(df["date"])

df = df.sort_values("date")

df.set_index("date", inplace=True)

In [4]:
# ==========================================================
# 3. SILICON GENOME HEADER
# ==========================================================

def genome_header(title):

    display(HTML(f"""
    <div style="
        background:
        linear-gradient(
        135deg,
        #050816,
        #101935,
        #1b4d91,
        #00d4ff
        );
        padding:28px;
        border-radius:24px;
        margin-top:20px;
        margin-bottom:15px;
        box-shadow:0px 0px 40px rgba(0,212,255,0.35);
    ">
        <h1 style="
        color:white;
        text-align:center;
        font-size:38px;
        font-family:Trebuchet MS;
        letter-spacing:3px;">
        {title}
        </h1>
    </div>
    """))

genome_header("🧬 ARM Silicon Genome Research Facility")


In [5]:
# ==========================================================
# 4. CORE METABOLISM
# ==========================================================

genome_header("⚛️ Core Market Metabolism")

df["returns"] = df["close"].pct_change()

cagr = (
    (
        df["close"].iloc[-1]
        /
        df["close"].iloc[0]
    )
    **
    (252/len(df))
    - 1
) * 100

annual_vol = (
    df["returns"].std()
    * np.sqrt(252)
    * 100
)

max_drawdown = (
    (
        df["close"]
        /
        df["close"].cummax()
    )
    - 1
).min() * 100

metrics = pd.DataFrame({

    "Metric":[
        "CAGR %",
        "Annual Volatility %",
        "Max Drawdown %"
    ],

    "Value":[
        round(cagr,2),
        round(annual_vol,2),
        round(max_drawdown,2)
    ]
})

fig = px.funnel(
    metrics,
    x="Value",
    y="Metric",
    title="Silicon Vital Statistics"
)

fig.show()


In [6]:
# ==========================================================
# 5. INSTRUCTION FLOW ENGINE
# ==========================================================

genome_header("🔷 Instruction Flow Engine")

df["flow_7"] = df["close"].pct_change(7)

df["flow_30"] = df["close"].pct_change(30)

df["flow_90"] = df["close"].pct_change(90)

df["instruction_flow"] = (
    df["flow_7"] * 0.45
    +
    df["flow_30"] * 0.35
    +
    df["flow_90"] * 0.20
)

fig = px.area(
    df,
    y="instruction_flow",
    title="Instruction Flow Momentum"
)

fig.show()


In [7]:
# ==========================================================
# 6. CHIP DEMAND PRESSURE
# ==========================================================

genome_header("📡 Chip Demand Pressure Scanner")

df["volume_force"] = (
    np.log1p(df["volume"])
)

df["price_force"] = (
    abs(df["returns"])
)

df["chip_pressure"] = (
    df["volume_force"]
    *
    df["price_force"]
)

fig = px.line(
    df,
    y="chip_pressure",
    title="Chip Demand Pressure"
)

fig.show()


In [8]:
# ==========================================================
# 7. DESIGN EVOLUTION MAP
# ==========================================================

genome_header("🧠 Architecture Evolution Map")

df["ema21"] = df["close"].ewm(span=21).mean()

df["ema55"] = df["close"].ewm(span=55).mean()

df["evolution_gap"] = (
    df["ema21"]
    -
    df["ema55"]
)

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["evolution_gap"],
        fill="tozeroy",
        name="Evolution Gap"
    )
)

fig.update_layout(
    title="Architecture Evolution Cycle"
)

fig.show()


In [9]:
# ==========================================================
# 8. SEMICONDUCTOR DNA SCORE
# ==========================================================

genome_header("🧬 Semiconductor DNA Score")

vol_rank = (
    df["volume"]
    .rolling(30)
    .mean()
    .rank(pct=True)
)

trend_rank = (
    (
        df["close"]
        /
        df["close"].rolling(100).mean()
    )
    .rank(pct=True)
)

strength_rank = (
    df["returns"]
    .rolling(30)
    .mean()
    .rank(pct=True)
)

df["dna_score"] = (
    vol_rank * 0.30
    +
    trend_rank * 0.40
    +
    strength_rank * 0.30
)

fig = px.line(
    df,
    y="dna_score",
    title="Semiconductor DNA Strength"
)

fig.show()


In [10]:
# ==========================================================
# 9. SILICON CLIMATE MODEL
# ==========================================================

genome_header("🌍 Silicon Climate Model")

volatility = (
    df["returns"]
    .rolling(20)
    .std()
)

momentum = (
    df["close"]
    .pct_change(20)
)

conditions = [

    (momentum > 0.10),

    (momentum > 0) & (momentum <= 0.10),

    (momentum < 0) & (volatility < volatility.median()),

    (momentum < 0) & (volatility > volatility.median())

]

labels = [

    "Hyper Growth",

    "Expansion",

    "Optimization",

    "Thermal Shock"

]

df["climate"] = np.select(
    conditions,
    labels,
    default="Neutral"
)

fig = px.scatter(
    df,
    x=df.index,
    y="close",
    color="climate",
    title="Silicon Climate Regimes"
)

fig.show()


In [11]:
# ==========================================================
# 10. PROCESS NODE CLUSTERING
# ==========================================================

genome_header("⚙️ Process Node Intelligence")

cluster_data = df[
    [
        "instruction_flow",
        "chip_pressure",
        "dna_score"
    ]
].fillna(0)

scaler = RobustScaler()

scaled = scaler.fit_transform(
    cluster_data
)

model = KMeans(
    n_clusters=6,
    random_state=42,
    n_init=10
)

df["node_cluster"] = model.fit_predict(
    scaled
)

fig = px.scatter(
    df,
    x=df.index,
    y="close",
    color=df["node_cluster"].astype(str),
    title="Process Node Clusters"
)

fig.show()


In [12]:
# ==========================================================
# 11. NEURAL CORE ANALYZER
# ==========================================================

genome_header("🛰️ Neural Core Analyzer")

pca = PCA(n_components=2)

components = pca.fit_transform(
    scaled
)

pca_df = pd.DataFrame(
    components,
    columns=["PC1","PC2"]
)

fig = px.scatter(
    pca_df,
    x="PC1",
    y="PC2",
    title="Neural Core Feature Space"
)

fig.show()


In [13]:
# ==========================================================
# 12. SILICON ACCELERATION ZONES
# ==========================================================

genome_header("🚀 Silicon Acceleration Zones")

df["acceleration"] = (
    df["instruction_flow"]
    -
    df["instruction_flow"].rolling(20).mean()
)

signal = (

    (df["acceleration"] > 0)

    &

    (df["dna_score"] >
     df["dna_score"].rolling(50).mean())

)

df["signal"] = signal.astype(int)

fig = go.Figure()

fig.add_trace(

    go.Scatter(
        x=df.index,
        y=df["close"],
        name="ARM Price"
    )
)

fig.add_trace(

    go.Scatter(
        x=df.index[df["signal"] == 1],
        y=df["close"][df["signal"] == 1],
        mode="markers",
        name="Acceleration Zone"
    )
)

fig.update_layout(
    title="Silicon Acceleration Detection"
)

fig.show()


In [14]:
# ==========================================================
# 13. GENOME INTELLIGENCE REPORT
# ==========================================================

genome_header("📘 Genome Intelligence Report")

print("""

1. Instruction Flow measures multi-horizon momentum.

2. Chip Pressure identifies demand concentration.

3. DNA Score combines trend, strength and liquidity.

4. Silicon Climate classifies market environments.

5. Process Node Clustering discovers hidden structures.

6. Neural Core Analysis compresses market behavior.

7. Acceleration Zones identify potential growth phases.

8. ARM behaves like a living semiconductor ecosystem.

""")



1. Instruction Flow measures multi-horizon momentum.

2. Chip Pressure identifies demand concentration.

3. DNA Score combines trend, strength and liquidity.

4. Silicon Climate classifies market environments.

5. Process Node Clustering discovers hidden structures.

6. Neural Core Analysis compresses market behavior.

7. Acceleration Zones identify potential growth phases.

8. ARM behaves like a living semiconductor ecosystem.


